[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivamkalakaar/local-ai-lab-agents/blob/main/notebooks/00_setup_first_call.ipynb)

# 00 · Setup and your first local model call
**Local AI Lab · Build and run agents on your own computer**

This course runs every agent on a model that lives on this machine. No API keys, no cloud AI, no data leaving the box.

**Before you start (Colab):** `Runtime → Change runtime type → T4 GPU`. Then run the setup cell below. It installs [Ollama](https://ollama.com), starts it, and downloads Meta's Llama 3.1 8B (about 5 GB). It takes 2–3 minutes the first time.

Companion reading: [Agent Engineering School, Module 00](https://claude.ai/artifact/41sHaxGwFWf7SjJWDdtK5w#m0)

In [1]:
# Local AI Lab setup: run this cell first (about 2-3 minutes on Colab, a few seconds locally)
import os, sys, subprocess
EXTRA = []
if "google.colab" in sys.modules:
    if not os.path.exists("/content/local-ai-lab-agents"):
        subprocess.run(["git", "clone", "-q", "https://github.com/shivamkalakaar/local-ai-lab-agents", "/content/local-ai-lab-agents"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/local-ai-lab-agents", "pull", "-q"])   # get the latest fixes
    os.chdir("/content/local-ai-lab-agents")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", *EXTRA], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "projects", "cs329a", "cs329a_projects"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from lal.runtime import setup
MODEL = setup()

Installing Ollama (about 1 minute)...
Ready in 202s. Main model: llama3.1:8b. GPU: yes.
Model server: http://localhost:11434/v1 (on this machine). No API keys, no data leaves it.


## 1. Talk to the model on this machine
Ollama runs the model on this computer and serves it at `http://localhost:11434`. It speaks the same API as OpenAI, so we use the normal `openai` Python library and only change two things:
- `base_url` points at **localhost**, not at OpenAI's servers;
- the API key is a dummy value, because nothing leaves this machine.

The setup cell already loaded the model into GPU memory (that first load takes about a minute), so this call should take a second or two.

In [2]:
import time
from openai import OpenAI

local = OpenAI(base_url="http://localhost:11434/v1", api_key="not-needed")   # the model server on THIS machine
t0 = time.time()
resp = local.chat.completions.create(
    model=MODEL,                                           # llama3.1:8b, downloaded by the setup cell
    messages=[{"role": "user", "content": "In one sentence: why would a law firm want AI that runs on its own computers?"}],
    temperature=0,
)
reply = resp.choices[0].message.content
print(reply)
print(f"\n{time.time() - t0:.1f} seconds, model = {MODEL}, server = {local.base_url}")

A law firm would want AI that runs on its own computers to improve efficiency, accuracy, and scalability in tasks such as document review, contract analysis, and research, while also maintaining control over sensitive client data and intellectual property.

4.0 seconds, model = llama3.1:8b, server = http://localhost:11434/v1/


Every later notebook uses two short helpers from `lal/llm.py`, so the code stays readable. They wrap exactly the call above, pointed at the same local server:
- `ask(prompt)`: prompt in, text out;
- `structured(prompt, Schema)`: forced JSON that matches a Pydantic schema.

In [3]:
from lal.llm import ask, chat, client
print("lal.llm talks to:", client.base_url)            # the same localhost address
print(ask("Reply with exactly: ready"))

lal.llm talks to: http://localhost:11434/v1/
Ready


## 2. Measure speed and token use
On your own hardware there is no per-token bill, but speed still matters.

In [4]:
t0 = time.time()
resp = local.chat.completions.create(model=MODEL, temperature=0,
    messages=[{"role": "user", "content": "List five common clauses in a services contract, one line each."}])
secs = time.time() - t0
u = resp.usage
print(resp.choices[0].message.content)
print(f"\nprompt tokens: {u.prompt_tokens}, output tokens: {u.completion_tokens}, "
      f"speed: {u.completion_tokens / secs:.0f} tokens/second")

Here are five common clauses in a services contract:

1. **Scope of Work**: The services to be provided by the contractor are outlined in detail.
2. **Payment Terms**: The method and timing of payment for services rendered are specified.
3. **Intellectual Property Rights**: Ownership and use of intellectual property created during the contract are defined.
4. **Confidentiality**: The contractor agrees to keep confidential any sensitive information shared by the client.
5. **Termination**: The conditions under which the contract can be terminated by either party are outlined.

prompt tokens: 23, output tokens: 112, speed: 38 tokens/second


## 3. Temperature: same question, different settings
Low temperature gives the same answer every time (good for extraction). Higher temperature gives variety (good for brainstorming, or for sampling several candidates to check).

In [5]:
for temp in (0, 1.0):
    answers = {ask("Name one risky clause in a vendor contract. Answer in 3 words or fewer.", temperature=temp).strip()
               for _ in range(3)}
    print(f"temperature {temp}: {answers}")

temperature 0: {'Indemnity clause.'}
temperature 1.0: {'Liquidity lockup.', 'Arbitration over litigation.', 'Liquidated damages.'}


## 4. Our practice data
We use 20 made-up service agreements. Some have problems planted on purpose: an uncapped or oversized liability cap, the wrong governing law, a long auto-renewal, or a short notice period. Because we planted them, we know the right answers, so we can grade our agents. (See `data/gen.py`.)

In [6]:
from lal.data import contract, labels, playbook, ids
print(contract("msa_07")[:900], "...")
print("\nTrue answer for msa_07:", labels()["msa_07"])
print("\nPlaybook rules:")
for k, v in playbook()["rules"].items():
    print(f"  - {k}: {v}")

MASTER SERVICES AGREEMENT

This Master Services Agreement (the "Agreement") is entered into by Harbor Point Logistics ("Customer") and Orbit Software LLC ("Vendor").

1. SERVICES
1.1 Vendor shall provide the document management and support services described in Exhibit A (the "Services") to Customer.

2. FEES
2.1 Customer shall pay Vendor $4,500 per month, invoiced monthly and due within thirty (30) days.

3. TERM
3.1 This Agreement begins on January 1, 2027 and continues for an initial term of thirty-six (36) months.

4. RENEWAL
4.1 After the initial term, this Agreement shall automatically renew for successive one (1) year periods unless either party gives at least 90 days' prior written notice of non-renewal.

5. CONFIDENTIALITY
5.1 Each party shall keep the other party's Confidential Information secret and use it only to perform this Agreement.

6. DATA PROTECTION
6.1 Vendor shall pr ...

True answer for msa_07: {'id': 'msa_07', 'terms': {'governing_law': 'Delaware', 'liability_cap

In [7]:
from lal.checks import check
check("model answers", len(reply) > 10)
check("20 practice contracts loaded", len(ids()) == 20)

PASS: model answers
PASS: 20 practice contracts loaded


True

## Your turn
1. Ask the model to summarize `msa_07` in three bullet points. Time it.
2. Change `temperature` and see how the summary changes.

**Next:** 01 · Build an agent loop from scratch

## Solutions
Try the exercises yourself first, then compare.

In [8]:
# Solution 1: summarize msa_07 in three bullets and time it
t0 = time.time()
summary = ask("Summarize this contract in exactly three short bullet points:\n\n" + contract("msa_07"))
print(summary)
print(f"\n{time.time() - t0:.1f} seconds")

Here are three short bullet points summarizing the contract:

* The Vendor, Orbit Software LLC, will provide document management and support services to the Customer, Harbor Point Logistics, for $4,500 per month.
* The contract will automatically renew for one-year periods after an initial 36-month term, unless either party gives 90 days' written notice of non-renewal.
* The Vendor's total liability under the contract will not exceed the fees paid by the Customer in the 24 months preceding a claim.

3.1 seconds


In [9]:
# Solution 2: the same summary at three temperatures
for temp in (0, 0.7, 1.3):
    print(f"--- temperature {temp} ---")
    print(ask("Summarize this contract in exactly three short bullet points:\n\n" + contract("msa_07"), temperature=temp))
# What to notice: at 0 the summary is the same every run. At higher temperatures the wording changes,
# and at very high values facts can drift (for example, the cap period). Use 0 for anything you will check.

--- temperature 0 ---
Here are three short bullet points summarizing the contract:

* The Vendor, Orbit Software LLC, will provide document management and support services to the Customer, Harbor Point Logistics, for $4,500 per month.
* The contract will automatically renew for one-year periods after an initial 36-month term, unless either party gives 90 days' written notice of non-renewal.
* The Vendor's total liability under the contract will not exceed the fees paid by the Customer in the 24 months preceding a claim.
--- temperature 0.7 ---
Here are three short bullet points summarizing the contract:

• **Services and Fees**: Vendor will provide document management and support services to Customer for $4,500 per month, invoiced monthly and due within 30 days.
• **Term and Renewal**: The initial term is 36 months, after which the agreement will automatically renew for 1-year periods unless either party gives 90 days' prior written notice of non-renewal.
• **Liability and Termination*